# DMPSP — Training on Kaggle (P100 16GB)

Trains all three DMPSP components on Kaggle's free P100 GPU (30h/week).

**Timeout-safe**: checkpoints saved every 2500 steps. If session dies, re-run from the
same cell — `--resume` picks up the latest checkpoint automatically.

**Setup** (one-time):
1. Fork this notebook on Kaggle
2. Enable GPU: Settings → Accelerator → GPU P100
3. Upload your `data/processed/` folder as a Kaggle Dataset input named `dmpsp-processed-data`
   - OR leave it empty: the notebook auto-downloads USPTO-50K (~50K reactions)
4. Clone or upload the DMPSP repo to `/kaggle/working/dmpsp/`

**Expected P100 training times** (100K steps each):
- Action proposal: ~3-4 hours (batch=64)
- World model: ~8-10 hours (batch=16, ReactionT5 fine-tuning)
- Value function: ~2-3 hours (batch=64)

Split across 2-3 sessions using `--resume`. Kaggle gives 30h/week free.

In [ ]:
# Cell 1: Install dependencies
# Run once per session. Takes ~3-5 minutes.
import subprocess, sys

# torch + torch-geometric with CUDA (Kaggle has CUDA 12.x)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'torch==2.4.0', 'torchvision==0.19.0',
    '--index-url', 'https://download.pytorch.org/whl/cu121',
], check=True)

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'torch-geometric==2.4.0',
    'torch-scatter==2.1.2',
    'torch-sparse==0.6.18',
    '-f', 'https://data.pyg.org/whl/torch-2.4.0+cu121.html',
], check=True)

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.44.0',
    'accelerate==0.33.0',
    'sentencepiece==0.2.0',
    'rdkit==2024.03.5',
    'httpx==0.27.2',
    'PyYAML==6.0.2',
    'pandas==2.2.2',
    'pyarrow==17.0.0',
], check=True)

print('Dependencies installed.')

In [ ]:
# Cell 2: Paths and GPU check
import os, sys, torch
from pathlib import Path

PROJECT_ROOT = Path('/kaggle/working/dmpsp')
sys.path.insert(0, str(PROJECT_ROOT))

# Checkpoints persist in Kaggle output across sessions
CHECKPOINT_DIR = Path('/kaggle/working/checkpoints')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Data: uploaded dataset or auto-downloaded
UPLOADED_DATA = Path('/kaggle/input/dmpsp-processed-data')
DATA_DIR = UPLOADED_DATA if UPLOADED_DATA.exists() else Path('/kaggle/working/data/processed')

LOG_DIR = Path('/kaggle/working/logs')
LOG_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root : {PROJECT_ROOT}')
print(f'Data dir     : {DATA_DIR}  (exists={DATA_DIR.exists()})')
print(f'Checkpoint dir: {CHECKPOINT_DIR}')
print(f'GPU available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU           : {torch.cuda.get_device_name(0)}')
    print(f'VRAM          : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Cell 3: Prepare data (skip if already present)
# Auto-downloads USPTO-50K from HuggingFace if DATA_DIR is empty.
# Uses subprocess to avoid pyarrow/torch DLL conflict.

train_traj = DATA_DIR / 'trajectories_train.pkl'

if train_traj.exists():
    print(f'Data already present: {DATA_DIR}')
else:
    print('Downloading and preprocessing USPTO-50K ...')
    import subprocess, sys
    result = subprocess.run(
        [
            sys.executable,
            str(PROJECT_ROOT / 'scripts' / 'prepare_data.py'),
            '--source', 'uspto50k',
            '--out_dir', str(DATA_DIR),
        ],
        capture_output=False,   # stream output live
    )
    if result.returncode != 0:
        raise RuntimeError('prepare_data.py failed')

# Verify
import pickle
with open(DATA_DIR / 'trajectories_train.pkl', 'rb') as f:
    trajs = pickle.load(f)
print(f'Train trajectories: {len(trajs)}')
print(f'Val trajectories  : {len(pickle.load(open(DATA_DIR / "trajectories_val.pkl", "rb")))}')

## Phase 1 — Train Action Proposal Diffusion

Fastest model. ~3-4h on P100. `--resume` auto-continues from latest checkpoint.

In [ ]:
# Cell 4: Train Action Proposal (100K steps, batch=64)
# Re-run this cell to resume after session timeout.
import subprocess, sys
from pathlib import Path

PROJECT_ROOT = Path('/kaggle/working/dmpsp')
DATA_DIR = Path('/kaggle/input/dmpsp-processed-data') if Path('/kaggle/input/dmpsp-processed-data').exists() \
           else Path('/kaggle/working/data/processed')
CHECKPOINT_DIR = Path('/kaggle/working/checkpoints')

cmd = [
    sys.executable,
    str(PROJECT_ROOT / 'train' / 'train_proposal.py'),
    '--model_config', str(PROJECT_ROOT / 'configs' / 'model.yaml'),
    '--train_config', str(PROJECT_ROOT / 'configs' / 'train.yaml'),
    '--data_dir', str(DATA_DIR),
    '--out_dir', str(CHECKPOINT_DIR / 'action_proposal'),
    '--device', 'cuda',
    '--batch_size', '64',
    '--max_steps', '100000',
    '--save_every', '2500',
    '--log_level', 'INFO',
    '--resume',
]
print('Command:', ' '.join(cmd))
result = subprocess.run(cmd)
print(f'Exit code: {result.returncode}')

## Phase 2 — Train World Model (ReactionT5 fine-tuning)

Heaviest step — ReactionT5 + property heads + DynamicsDiffusion.
Gradient checkpointing enabled for P100 (16GB). ~8-10h total across 1-2 sessions.

In [ ]:
# Cell 5: Train World Model (100K steps, batch=16)
# Re-run to resume after timeout.
import subprocess, sys
from pathlib import Path

PROJECT_ROOT = Path('/kaggle/working/dmpsp')
DATA_DIR = Path('/kaggle/input/dmpsp-processed-data') if Path('/kaggle/input/dmpsp-processed-data').exists() \
           else Path('/kaggle/working/data/processed')
CHECKPOINT_DIR = Path('/kaggle/working/checkpoints')

cmd = [
    sys.executable,
    str(PROJECT_ROOT / 'train' / 'train_world_model.py'),
    '--model_config', str(PROJECT_ROOT / 'configs' / 'model.yaml'),
    '--train_config', str(PROJECT_ROOT / 'configs' / 'train.yaml'),
    '--data_dir', str(DATA_DIR),
    '--out_dir', str(CHECKPOINT_DIR / 'world_model'),
    '--device', 'cuda',
    '--batch_size', '16',
    '--max_steps', '100000',
    '--save_every', '2500',
    '--log_level', 'INFO',
    '--resume',
]
print('Command:', ' '.join(cmd))
result = subprocess.run(cmd)
print(f'Exit code: {result.returncode}')

## Phase 3 — Train Value Function

Fastest phase. ~2-3h on P100. 10-head Transformer (10 synthesis objectives).

In [ ]:
# Cell 6: Train Value Function (100K steps, batch=64)
import subprocess, sys
from pathlib import Path

PROJECT_ROOT = Path('/kaggle/working/dmpsp')
DATA_DIR = Path('/kaggle/input/dmpsp-processed-data') if Path('/kaggle/input/dmpsp-processed-data').exists() \
           else Path('/kaggle/working/data/processed')
CHECKPOINT_DIR = Path('/kaggle/working/checkpoints')

cmd = [
    sys.executable,
    str(PROJECT_ROOT / 'train' / 'train_value.py'),
    '--model_config', str(PROJECT_ROOT / 'configs' / 'model.yaml'),
    '--train_config', str(PROJECT_ROOT / 'configs' / 'train.yaml'),
    '--data_dir', str(DATA_DIR),
    '--out_dir', str(CHECKPOINT_DIR / 'value_fn'),
    '--device', 'cuda',
    '--batch_size', '64',
    '--max_steps', '100000',
    '--save_every', '2500',
    '--log_level', 'INFO',
    '--resume',
]
print('Command:', ' '.join(cmd))
result = subprocess.run(cmd)
print(f'Exit code: {result.returncode}')

## Checkpoint Status

In [ ]:
# Cell 7: Show checkpoint status for all three models
from pathlib import Path

CHECKPOINT_DIR = Path('/kaggle/working/checkpoints')

for model_name in ['action_proposal', 'world_model', 'value_fn']:
    ckpt_dir = CHECKPOINT_DIR / model_name
    if not ckpt_dir.exists():
        print(f'{model_name}: NOT STARTED')
        continue
    ckpts = sorted(ckpt_dir.glob('checkpoint_step*.pt'))
    if not ckpts:
        print(f'{model_name}: directory exists but no checkpoints')
    else:
        latest = ckpts[-1]
        # Parse step number from filename
        step = int(latest.stem.replace('checkpoint_step', ''))
        size_mb = latest.stat().st_size / 1e6
        print(f'{model_name}: step {step:,} / 100000  ({size_mb:.0f} MB)  [{len(ckpts)} checkpoints]')

## Smoke Test

In [ ]:
# Cell 8: Smoke test — plan a route for aspirin
import json, subprocess, sys
from pathlib import Path

PROJECT_ROOT = Path('/kaggle/working/dmpsp')
CHECKPOINT_DIR = Path('/kaggle/working/checkpoints')

result = subprocess.run(
    [
        sys.executable,
        str(PROJECT_ROOT / 'scripts' / 'plan_route.py'),
        '--smiles', 'CC(=O)Oc1ccccc1C(=O)O',
        '--checkpoint_dir', str(CHECKPOINT_DIR),
        '--device', 'cuda',
        '--max_steps', '3',
        '--log_level', 'WARNING',
    ],
    capture_output=True, text=True,
)

if result.returncode == 0:
    route = json.loads(result.stdout)
    print(f"Route: {route['n_steps']} steps")
    print(f"Yield: {route['total_yield_fraction']:.3f}")
    print(f"Objective scores: {route['objective_scores']}")
    print(f"Planning time: {route['planning_time_seconds']:.2f}s")
else:
    print('plan_route.py failed:')
    print(result.stderr)

## Benchmark (optional — after all 3 models trained)

In [ ]:
# Cell 9: Run benchmark comparison (beam vs MCTS vs random, 100 test molecules)
import subprocess, sys
from pathlib import Path

PROJECT_ROOT = Path('/kaggle/working/dmpsp')
CHECKPOINT_DIR = Path('/kaggle/working/checkpoints')
DATA_DIR = Path('/kaggle/input/dmpsp-processed-data') if Path('/kaggle/input/dmpsp-processed-data').exists() \
           else Path('/kaggle/working/data/processed')

result = subprocess.run(
    [
        sys.executable,
        str(PROJECT_ROOT / 'scripts' / 'benchmark.py'),
        '--checkpoint_dir', str(CHECKPOINT_DIR),
        '--data_dir', str(DATA_DIR),
        '--device', 'cuda',
        '--n_molecules', '100',
        '--max_steps', '5',
        '--out_csv', '/kaggle/working/results/benchmark.csv',
        '--out_json', '/kaggle/working/results/benchmark.json',
    ],
)

## Save & Download Checkpoints

Kaggle outputs persist at `/kaggle/working/` after session ends and are downloadable
from **Output → Files**. Zip them for convenience.

In [ ]:
# Cell 10: Zip latest checkpoints for download
import shutil
from pathlib import Path

CHECKPOINT_DIR = Path('/kaggle/working/checkpoints')
zip_path = '/kaggle/working/dmpsp_checkpoints'

shutil.make_archive(zip_path, 'zip', str(CHECKPOINT_DIR))
zip_size = Path(zip_path + '.zip').stat().st_size / 1e6
print(f'Zipped: {zip_path}.zip  ({zip_size:.0f} MB)')
print('Download: Kaggle notebook → Output tab → Files → dmpsp_checkpoints.zip')